# 04 — Trực quan hóa kết quả thực nghiệm

Notebook đọc manifest, đầu ra inference và điểm đã lưu bởi notebook 03. Sau khi chạy cell lưu điểm trong notebook 03, thêm đường dẫn các thư mục lần chạy vào `RUN_DIRS`. Dùng `EVALUATION_FILES` nếu lưu điểm với tên khác. Chọn tối đa một lần chạy cho mỗi model và dùng cùng dataset, cùng số lần sinh mỗi câu. Các cấu hình khác nhau vẫn được hiển thị trong bảng tổng hợp để giới hạn cách diễn giải.

Mở notebook từ gốc bản sao dự án hoặc thư mục `notebooks/`. Khi chưa chọn lần chạy, notebook không tạo biểu đồ.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import runpy

cwd = Path.cwd()
helper = next((base / 'src/r1_project_root.py' for base in (cwd, *cwd.parents)
               if (base / 'src/r1_project_root.py').is_file()), None)
if helper is None:
    matches = list(cwd.glob('*/src/r1_project_root.py'))
    if len(matches) != 1:
        raise FileNotFoundError('Không xác định được repository. Hãy clone dự án hoặc chuyển cwd vào bản clone.')
    helper = matches[0]
project_root = runpy.run_path(str(helper))['find_project_root']()
sys.path.insert(0, str(project_root))
root = project_root
from src.dataset import DIFFICULTIES
from src.evaluation import (
    accuracy_metrics, language_analysis, length_accuracy_relationship, reasoning_length_metrics,
)
from src.evaluation_artifacts import load_saved_evaluation

RUN_DIRS = []
EVALUATION_FILES = {}  # Khóa là mục trong RUN_DIRS, giá trị là tên tệp trong thư mục lần chạy.
runs = {}
dataset_digest = None
repeat_count = None
for run_dir in RUN_DIRS:
    directory = root / run_dir
    evaluation_path = directory / EVALUATION_FILES.get(run_dir, 'evaluation.json')
    manifest, questions, rows = load_saved_evaluation(directory, evaluation_path)
    digest = manifest['dataset_sha256']
    repeats = len(manifest['inference']['seeds'])
    if runs and (dataset_digest != digest or repeat_count != repeats):
        raise ValueError('Chọn các lần chạy có cùng dataset và số lần sinh mỗi câu.')
    dataset_digest, repeat_count = digest, repeats
    model = manifest['model']['id'].split('/')[-1].removeprefix('DeepSeek-R1-Distill-')
    if model in runs:
        raise ValueError(f'Chọn một lần chạy cho {model}.')
    runs[model] = (manifest, questions, rows)

if not runs:
    print('Chưa chọn kết quả thực nghiệm. Điền RUN_DIRS sau khi lưu điểm ở notebook 03.')
colors = dict(zip(runs, plt.get_cmap('tab10').colors))

## Accuracy theo mô hình, chủ đề và độ khó

Dùng cùng định nghĩa accuracy với notebook 03. Mỗi cột là tỷ lệ trả lời đúng trong một lần sinh trên toàn bộ lượt dự kiến của nhóm. Cột trống nghĩa là nhóm còn lượt thiếu hoặc chờ chấm. So sánh nhóm nhỏ cần xem cả cỡ mẫu và số lượt chờ trong bảng.

In [ ]:
summary_rows = []
group_rows = []
for model, (manifest, questions, rows) in runs.items():
    overall = accuracy_metrics(rows, len(questions) * repeat_count)
    lengths = reasoning_length_metrics(rows)
    languages = language_analysis(rows)
    options = manifest['inference']
    summary_rows.append({
        'Model': model, 'Run ID': manifest['run_id'], **overall,
        'Token sinh TB': lengths['generated_token_count']['mean'],
        'Token suy luận TB (ước lượng)': lengths['reasoning_token_count']['mean'],
        'Lượt đo được token suy luận': lengths['reasoning_token_count']['n'],
        'Tỷ lệ trộn Việt–Anh': languages['mixed_anywhere_share_all'],
        'Trần token': options['generation']['max_new_tokens'],
        'Kiểu số': options['dtype'], 'Lượng tử hóa': options['quantization'] or 'Không',
        'Phiên bản prompt': manifest['prompt'].get('template_version'),
        'Revision model': manifest['model']['model_revision'],
    })
    for field in ('topic', 'difficulty'):
        for label in sorted({item[field] for item in questions}):
            expected = repeat_count * sum(item[field] == label for item in questions)
            subset = [row for row in rows if row[field] == label]
            group_rows.append({'Model': model, 'field': field, 'label': label,
                               **accuracy_metrics(subset, expected)})

summary = pd.DataFrame(summary_rows)
groups = pd.DataFrame(group_rows)
if runs:
    if summary['accuracy'].notna().any():
        fig, ax = plt.subplots(figsize=(8, 4))
        models = list(runs)
        values = pd.to_numeric(summary.set_index('Model')['accuracy']).reindex(models)
        ax.bar(models, values, color=[colors[model] for model in models])
        ax.set(title='Accuracy theo mô hình', xlabel='Checkpoint Distill-Qwen',
               ylabel='Tỷ lệ trả lời đúng', ylim=(0, 1.05))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        fig.tight_layout()
        plt.show()
    else:
        print('Chưa có accuracy tổng thể đã chốt.')

    for field, label in [('topic', 'Chủ đề'), ('difficulty', 'Độ khó')]:
        by_group = groups.loc[groups['field'].eq(field)]
        display(by_group[['Model', 'label', 'expected', 'observed', 'pending_review', 'accuracy']])
        pivot = by_group.pivot(index='label', columns='Model', values='accuracy').reindex(columns=runs)
        if field == 'difficulty':
            pivot = pivot.reindex(DIFFICULTIES)
        pivot = pivot.apply(pd.to_numeric)
        if not pivot.notna().any().any():
            print(f'Chưa có accuracy đã chốt theo {label.lower()}.')
            continue
        # pandas có thể hiển thị NaN như cột 0. Vẽ trực tiếp để giữ vị trí trống.
        fig, ax = plt.subplots(figsize=(9, 4))
        positions = np.arange(len(pivot))
        width = 0.8 / len(runs)
        for index, model in enumerate(runs):
            shift = (index - (len(runs) - 1) / 2) * width
            ax.bar(positions + shift, pivot[model], width, label=model, color=colors[model])
        ax.set_xticks(positions, labels=pivot.index, rotation=15, ha='right')
        ax.set(title=f'Accuracy theo {label.lower()}', xlabel=label,
               ylabel='Tỷ lệ trả lời đúng', ylim=(0, 1.05))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.legend(title='Model')
        fig.tight_layout()
        plt.show()

## Phân phối độ dài chuỗi suy luận

Độ dài ở đây là số token ước lượng bằng tokenizer tương ứng trên phần suy luận có ranh giới xác định. Các mô hình dùng cùng biên khoảng trên biểu đồ. Mẫu số trong chú giải là số lượt đo được, vì lượt thiếu thẻ đóng hoặc chưa mã hóa lại không có độ dài suy luận.

In [ ]:
reasoning = {
    model: [row['reasoning_token_count'] for row in rows if row['reasoning_token_count'] is not None]
    for model, (_, _, rows) in runs.items()
}
all_lengths = [length for values in reasoning.values() for length in values]
if all_lengths:
    bins = np.histogram_bin_edges(all_lengths, bins=20)
    fig, ax = plt.subplots(figsize=(9, 4))
    for model, values in reasoning.items():
        if values:
            ax.hist(values, bins=bins, histtype='step', linewidth=2, color=colors[model],
                    label=f'{model} (n={len(values)})')
    ax.set(title='Phân phối độ dài chuỗi suy luận', xlabel='Số token suy luận (ước lượng)',
           ylabel='Số lượt sinh')
    ax.legend()
    fig.tight_layout()
    plt.show()
elif runs:
    print('Chưa có độ dài suy luận. Kiểm tra tokenizer và độ phủ ranh giới ở notebook 03.')

## Tỷ lệ sử dụng ngôn ngữ

Hai biểu đồ dùng toàn bộ lượt quan sát được làm mẫu số và giữ nhóm không xác định. Nhãn trộn trong mỗi biểu đồ chỉ tính tín hiệu Việt–Anh xuất hiện trong cùng phần. Tỷ lệ trộn trong bảng tổng hợp còn tính trường hợp hai phần dùng hai ngôn ngữ khác nhau.

In [ ]:
labels = {'vi': 'Tiếng Việt', 'en': 'Tiếng Anh', 'mixed_vi_en': 'Trộn Việt–Anh',
          'unknown': 'Không xác định'}
available = {model: language_analysis(rows) for model, (_, _, rows) in runs.items() if rows}
if available:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, field, title in zip(axes, ('reasoning_language', 'final_language'),
                                ('Phần suy luận', 'Câu trả lời cuối')):
        shares = pd.DataFrame.from_dict(
            {model: stats[field]['share_all'] for model, stats in available.items()}, orient='index'
        ).rename(columns=labels)
        shares.plot.bar(stacked=True, ax=ax, rot=0,
                        color=['#087f8c', '#345995', '#e88e42', '#c5c5c5'])
        ax.set(title=title, xlabel='Model', ylabel='Tỷ lệ lượt sinh', ylim=(0, 1.05))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), fontsize=8)
    fig.suptitle('Phân phối ngôn ngữ theo phần đầu ra')
    fig.tight_layout()
    plt.show()
elif runs:
    print('Chưa có lượt sinh để tính tỷ lệ ngôn ngữ.')

## Số token và độ chính xác

Biểu đồ dùng tổng token đã sinh và tỷ lệ đúng trong từng khoảng token. Mọi mô hình dùng chung các khoảng, và nhãn `n` chỉ số lượt đã chấm trong mỗi điểm. Các lượt thiếu điểm bị loại khỏi phân tích mô tả này. Bảng tương quan point-biserial hiển thị kết quả cả khi giữ và khi loại các lượt chạm trần token. Độ khó và ngân sách sinh có thể ảnh hưởng đồng thời đến độ dài và khả năng trả lời đúng.

In [ ]:
relations = []
scored_rows = []
for model, (_, _, rows) in runs.items():
    for exclude in (False, True):
        relations.append({'Model': model, 'Bỏ lượt chạm trần': exclude,
                          **length_accuracy_relationship(rows, length_field='generated_token_count',
                                                         exclude_truncated=exclude)})
    scored_rows.extend({'Model': model, 'tokens': row['generated_token_count'],
                        'correct': row['correct']} for row in rows if row['correct'] is not None)

relation_table = pd.DataFrame(relations)
if runs:
    display(relation_table)
if scored_rows:
    scored = pd.DataFrame(scored_rows)
    edges = [0, 128, 256, 512, 1024, 2048, 4096, np.inf]
    ranges = ['0–127', '128–255', '256–511', '512–1023', '1024–2047', '2048–4095', '≥4096']
    scored['range'] = pd.cut(scored['tokens'], bins=edges, labels=ranges, right=False)
    by_tokens = scored.groupby(['Model', 'range'], observed=True)['correct'].agg(['mean', 'count'])
    fig, ax = plt.subplots(figsize=(10, 5))
    for model_index, model in enumerate(runs):
        if model not in by_tokens.index.get_level_values('Model'):
            continue
        values = by_tokens.loc[model].reindex(ranges)
        positions = np.arange(len(ranges)) + (model_index - (len(runs) - 1) / 2) * 0.13
        ax.scatter(positions, values['mean'], label=model, color=colors[model])
        for x, rate, n in zip(positions, values['mean'], values['count']):
            if pd.notna(rate):
                ax.annotate(f'n={int(n)}', (x, rate), xytext=(0, 6),
                            textcoords='offset points', ha='center', fontsize=8)
    ax.set_xticks(np.arange(len(ranges)), labels=ranges)
    ax.set(title='Tỷ lệ đúng quan sát theo khoảng token sinh', xlabel='Số token đã sinh',
           ylabel='Tỷ lệ đúng trong lượt đã chấm', ylim=(-0.05, 1.12))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.legend(title='Model')
    fig.tight_layout()
    plt.show()
elif runs:
    print('Chưa có lượt đã chấm để so sánh token với độ chính xác.')

## Bảng tổng hợp và phát hiện chính

Bảng tổng hợp liên kết model với mã lần chạy, revision, cấu hình và độ phủ chấm. Cận accuracy chỉ xét các lượt đã quan sát và thể hiện ảnh hưởng của các ca chờ chấm, không phải khoảng tin cậy.

In [ ]:
if runs:
    table = summary.copy()
    for column in ('accuracy', 'Tỷ lệ trộn Việt–Anh'):
        table[column] = table[column].map(lambda value: f'{value:.1%}' if pd.notna(value) else '—')
    display(table.round({'Token sinh TB': 1, 'Token suy luận TB (ước lượng)': 1}))
    findings = []
    for row in summary_rows:
        model = row['Model']
        accuracy = row['accuracy']
        if accuracy is None:
            findings.append(f"- **{model}** chưa có accuracy đã chốt. "
                            f"Còn {row['pending_review']} lượt chờ chấm và "
                            f"{row['missing_inference']} lượt chưa có đầu ra.")
        else:
            findings.append(f"- **{model}** đạt accuracy {accuracy:.1%} "
                            f"trên {row['observed']} lượt sinh.")
        if row['Tỷ lệ trộn Việt–Anh'] is not None:
            findings.append(f"- **{model}** có {row['Tỷ lệ trộn Việt–Anh']:.1%} lượt trộn Việt–Anh "
                            f"trong {row['observed']} lượt quan sát.")
    for item in relations:
        r = item['point_biserial_r']
        if r is not None and not item['Bỏ lượt chạm trần']:
            findings.append(f"- Tương quan token–độ chính xác của **{item['Model']}** "
                            f"là r = {r:.2f} trên {item['n']} lượt đã chấm.")
    display(Markdown('\n'.join(findings)))

## Hạn chế của thực nghiệm

- Đối chiếu phân tầng chủ đề và độ khó với số lượt dự kiến trước khi diễn giải chênh lệch. Các lượt sinh của cùng một câu có thể phụ thuộc nhau. Notebook này trình bày mô tả và chưa ước lượng khoảng tin cậy ghép cặp.
- So sánh checkpoint có thể chịu ảnh hưởng của lượng tử hóa, ngân sách token, prompt và môi trường GPU. Bảng tổng hợp cung cấp mã lần chạy để đối chiếu manifest.
- Ước lượng độ dài suy luận chỉ áp dụng cho các lượt có ranh giới xác định. Heuristic ngôn ngữ có thể bỏ sót phần trộn ngắn và nhầm thuật ngữ hoặc tên riêng.
- Tương quan token–độ chính xác không xác lập quan hệ nhân quả. Có thể quan sát tương quan do độ khó, đầu ra bị cắt hoặc do nhóm lượt được chấm có độ phủ khác nhau.